In [ ]:
from pyprojroot import here
import sys
sys.path.append(str(here() / 'methods'))
import pandas as pd
import numpy as np
from pyprojroot import here
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
from additional_utils.functions import coordinates2plot, mini_ranking_evaluations




processed_data = here('data/processed_data')
supplementary_information_data = here('data/processed_data/supplementary_data/')



# Supplementary Information B

In [ ]:
vars2keep = [
    'supplier_name_clean',
    'contract_price_mx',
    'rf_submission_period',
    'rf_decision_period',
    'rf_procedure_type',
    'rf_bl_conformity',
    'rf_buyer_dependence',
    'rf_single_bidder',
    'supplier_cri',
    'ncontracts_s',
    ]

contracts = pd.read_feather(processed_data / 'contracts_allfeatures.feather', columns=vars2keep)
contracts['prov_id'] = contracts.index

In [ ]:
sanctions_distance = pd.read_feather(processed_data / 'sanctions_distance.feather')

In [ ]:
sanctions_distance

In [ ]:
# suppliers that have contracts before or the same year than the sanction
s_before = sanctions_distance[sanctions_distance['difference'] <= 0]['supplier_name_clean'].unique()
# suppliers that have contracts after the sanction
s_after = sanctions_distance[sanctions_distance['difference'] > 0]['supplier_name_clean'].unique()
# intersection
s_returning = set(s_before).intersection(set(s_after))
# proportion of suppliers that gain contracts after the sanction
print(len(s_returning) / sanctions_distance['supplier_name_clean'].nunique())

In [ ]:
#contracts that hypothetically could be used as positive and negative
subset = sanctions_distance[sanctions_distance['supplier_name_clean'].isin(s_returning)]
ncontracts_subset = subset.shape[0]
print('total', ncontracts_subset )
print('positive', subset[subset['difference'] <= 0].shape[0], subset[subset['difference'] <= 0].shape[0] / ncontracts_subset )
print('negative', subset[subset['difference'] > 0 ].shape[0], subset[subset['difference'] > 0 ].shape[0] / ncontracts_subset )


In [ ]:
#suppliers that have contracts after a sanction that are sanctioned again
subset[subset['repeatedoffender'] ==1]['supplier_name_clean'].nunique() / len(s_returning)

## Supplementary Information B - HDSRF Model Evaluation

In [ ]:
files = list(supplementary_information_data.glob("LABELINGTEST*.json"))
all = []

for filename in files:  
  
    with open(filename, "r") as f:
        data = json.load(f)
    
    all = all + data

df = pd.DataFrame(all)


In [ ]:
#CSFull
average_gain_unCSFull_list = []
average_gain_calCSFull_list = []
average_lift_unCSFull_list = []
average_lift_calCSFull_list = []

for row in tqdm(range(len(df)), desc = 'Row'):
    ### CSFull
    y_true_CSFull = df.iloc[row]['y_test_CSFull']
    unCSFull_prob = df.iloc[row]['uncalibrated_probabilities_CSFull']
    calCSFull_prob = df.iloc[row]['calibrated_probabilities_CSFull']
    ### Uncalibrated CSFull
    average_gain_unCSFull, average_lift_unCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = unCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_unCSFull_list.append(average_gain_unCSFull) #Saving
    average_lift_unCSFull_list.append(average_lift_unCSFull) #Saving


    ### Calibrated CSFull
    average_gain_calCSFull, average_lift_calCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = calCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_calCSFull_list.append(average_gain_calCSFull) #Saving
    average_lift_calCSFull_list.append(average_lift_calCSFull) #Saving


#Include in dataset

#CSFull
df['avg_gain_unCSFull'] = average_gain_unCSFull_list
df['avg_gain_calCSFull'] = average_gain_calCSFull_list
df['avg_lift_unCSFull'] = average_lift_unCSFull_list
df['avg_lift_calCSFull'] = average_lift_calCSFull_list


In [ ]:
df.columns

In [ ]:
# Aggregate data
df2work = (
    df[['label2test', 'avg_gain_calCSFull', 'avg_lift_calCSFull']]
    .groupby('label2test')
    .agg(
        mean_gain=('avg_gain_calCSFull', 'mean'),
        std_gain=('avg_gain_calCSFull', 'std'),
        mean_lift=('avg_lift_calCSFull', 'mean'),
        std_lift=('avg_lift_calCSFull', 'std'),
    )
    .reset_index()
    .sort_values('mean_gain', ascending=True)
)

# Extract label parts
df2work[["assumption", "strategy", "time"]] = df2work["label2test"].str.extract(
    r"sanctioned([A-Z])_([A-Z])_(?:max)?(\d+|all)"
)


df2work["gain"] = df2work.apply(lambda x: f"{x['mean_gain']:.3f} ({x['std_gain']:.3f})", axis=1)
df2work["lift"] = df2work.apply(lambda x: f"{x['mean_lift']:.3f} ({x['std_lift']:.3f})", axis=1)

# keep only relevant columns (optional)
df2work = df2work.sort_values(by = 'mean_lift', ascending=False)[["assumption", "strategy", "time", "lift", "gain",]]


In [ ]:
#to latex
latex_str = df2work.to_latex(index=False)
print(latex_str)

## Supplementary Information B - One time offenders

In [ ]:
sanctions_distance = sanctions_distance.merge(contracts.drop(columns = 'supplier_name_clean'), on='prov_id', how='left')
sanctions_distance['before'] = np.where(sanctions_distance['difference'] <= 0, 1, 0)
sanctions_distance['afterORbefore'] = sanctions_distance.groupby(['supplier_name_clean'])['before'].transform('mean')

print(sanctions_distance.shape)
#this is to remove the supppliers that only have contracts before the sanctions
sanctions_distance = sanctions_distance[sanctions_distance['afterORbefore'] < 1].reset_index(drop=True)
print(sanctions_distance.shape)


print(sanctions_distance.shape)
#this is to remove the supppliers that only have contracts after the sanctions
sanctions_distance = sanctions_distance[sanctions_distance['afterORbefore'] > 0].reset_index(drop=True)
print(sanctions_distance.shape)

In [ ]:
sanctions_distance

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sanctions_distance_onetime = sanctions_distance[sanctions_distance['onetimeoffender'] == 1].reset_index(drop=True)
print(sanctions_distance_onetime.shape)




vars2remove = [
    'sanction_year',
    'difference',
    'onetimeoffender',
    'repeatedoffender',
    'afterORbefore'
]

sanctions_distance_onetime.drop(columns = vars2remove, inplace = True)


sanctions_distance_onetime['CRI'] = np.nanmean(sanctions_distance_onetime[[
    'rf_submission_period',
    'rf_decision_period',
    'rf_procedure_type',
    'rf_bl_conformity',
    'rf_buyer_dependence',
    'rf_single_bidder']], axis=1)

sanctions_distance_onetime['before'] = np.where(sanctions_distance_onetime['before'] == 1, 'before', 'after')    


df2plot = sanctions_distance_onetime.copy()
df2plot = df2plot.drop(columns = ['prov_id', 'contract_year']).groupby(['supplier_name_clean', 'before']).mean().reset_index()
df2plot.drop(columns = ['supplier_name_clean'], inplace = True)
df2plot['contract_price_mx'] = np.log(df2plot['contract_price_mx'])
df2plot['ncontracts_s'] = np.log(df2plot['ncontracts_s'])


# Exclude 'before' column from the features
feature_cols = [col for col in df2plot.columns if col not in ['sanction_months', 'before', 'ncontracts_s', 'CRI']]

# Create subplots
n_cols = 4  # number of plots per row
n_rows = -(-len(feature_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5)) #cols, rows
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(data=df2plot, x='before', y=col, ax=axes[i], order=['before', 'after'])
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Turn off empty subplots (if any)
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()

plt.show()

## Supplementary Information B - Repeated offenders

In [ ]:
# sanctions distance for repeated offenders
sanctions_distance_repeated = sanctions_distance[sanctions_distance['repeatedoffender'] == 1].reset_index(drop=True)
print(sanctions_distance_repeated.shape)

#positive and negative sanctions distance have a different treatment
sanctions_distance_repeated_pos = sanctions_distance_repeated[sanctions_distance_repeated['difference'] > 0].reset_index(drop=True)
print(sanctions_distance_repeated_pos.shape)
sanctions_distance_repeated_neg = sanctions_distance_repeated[sanctions_distance_repeated['difference'] < 0].reset_index(drop=True)
print(sanctions_distance_repeated_neg.shape)


#whith this every row will have a unique contract (prov_id)
idx = sanctions_distance_repeated_neg.groupby(['supplier_name_clean', 'contract_year', 'prov_id'])['difference'].idxmax()
sanctions_distance_repeated_neg = sanctions_distance_repeated_neg.loc[idx].reset_index(drop=True)


#whith this every row will have a unique contract (prov_id)
idx = sanctions_distance_repeated_pos.groupby(['supplier_name_clean', 'contract_year', 'prov_id'])['difference'].idxmin()
sanctions_distance_repeated_pos = sanctions_distance_repeated_pos.loc[idx].reset_index(drop=True)
#this will ensure that contracts that are before any sanction should not be included in the positive
sanctions_distance_repeated_pos = sanctions_distance_repeated_pos[~sanctions_distance_repeated_pos['prov_id'].isin(sanctions_distance_repeated_neg['prov_id'].unique())].reset_index(drop=True)


sanctions_distance_repeated = pd.concat([sanctions_distance_repeated_pos, sanctions_distance_repeated_neg], axis = 0, ignore_index = True)


vars2remove = [
    'sanction_year',
    'difference',
    'onetimeoffender',
    'repeatedoffender',
    'afterORbefore'
]

sanctions_distance_repeated.drop(columns = vars2remove, inplace = True)


sanctions_distance_repeated['CRI'] = np.nanmean(sanctions_distance_repeated[[
    'rf_submission_period',
    'rf_decision_period',
    'rf_procedure_type',
    'rf_bl_conformity',
    'rf_buyer_dependence',
    'rf_single_bidder']], axis=1)


sanctions_distance_repeated['before'] = np.where(sanctions_distance_repeated['before'] == 1, 'before', 'after') 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#df2plot = pd.concat([sanctions_distance_repeated, nonsanctioned], axis = 0, ignore_index = True)
df2plot = sanctions_distance_repeated.copy()
df2plot = df2plot.drop(columns = ['prov_id', 'contract_year']).groupby(['supplier_name_clean', 'before']).mean().reset_index()
df2plot.drop(columns = ['supplier_name_clean'], inplace = True)
df2plot['contract_price_mx'] = np.log(df2plot['contract_price_mx'])
df2plot['ncontracts_s'] = np.log(df2plot['ncontracts_s'])


# Exclude 'before' column from the features
feature_cols = [col for col in df2plot.columns if col not in ['before', 'sanction_months', 'ncontracts_s', 'CRI']]

# Create subplots
n_cols = 4  # number of plots per row
n_rows = -(-len(feature_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5)) #cols, rows
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(data=df2plot, x='before', y=col, ax=axes[i], order=['before', 'after'])
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Turn off empty subplots (if any)
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()

plt.show()

## Supplementary Information B - Mean Cosine distance of contracts of same sanctioned supplier across years

In [ ]:
#read
with open(supplementary_information_data / 'similarity4yearlabels.json', 'r') as f:
    similarity = json.load(f)

df = pd.DataFrame(similarity)

df = df[['year1', 'year2', 'cosine_mean', 'cosine_q50']]

In [ ]:
df2plot = df.groupby(['year1', 'year2']).mean().reset_index()

# 1. Create reversed pairs
df_reversed = df2plot.copy()
df_reversed.rename(columns={'year1': 'year2', 'year2': 'year1'}, inplace=True)

# 2. Concatenate original and reversed
df2plot = pd.concat([df2plot, df_reversed], ignore_index=True)

# 3. Add diagonal values with 0
years = sorted(set(df2plot['year1']).union(df2plot['year2']))
diagonal = pd.DataFrame({
    'year1': years,
    'year2': years,
    'cosine_mean': [0.0] * len(years)
})

# 4. Combine all
df2plot = pd.concat([df2plot, diagonal], ignore_index=True)

# 5. Pivot and plot
df2plot = df2plot.pivot(index='year1', columns='year2', values='cosine_mean')



# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df2plot, annot=True, 
            fmt=".2f", 
            cmap="coolwarm", 
            cbar_kws={'label': 'Mean Cosine Distance'},
            vmin=0, 
            vmax=0.5,)
#plt.title("Cosine Distance Heatmap")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Supplementary Information F: Details of train-test split

## Cumulative distribution of contracts among suppliers

In [ ]:
from scipy import integrate

# Contract accumulation
supplier_accumulation = contracts.groupby(['supplier_name_clean']).size().reset_index(name='ncontracts').sort_values(by='ncontracts', ascending=True).reset_index(drop=True)
supplier_accumulation['rank'] = supplier_accumulation.index + 1
supplier_accumulation['cummulative_rank'] = supplier_accumulation['rank'] / len(supplier_accumulation)
supplier_accumulation['cummulative_ncontracts'] = supplier_accumulation['ncontracts'].cumsum() / supplier_accumulation['ncontracts'].sum()
#supplier_accumulation['']
#gini coefficient
#Gini is equal to 1 - 2*area under the Lorenz curve (if axes run from 0 to 1)
gini = 1 - (2 * integrate.trapezoid(y = supplier_accumulation['cummulative_ncontracts'].values, 
                     x = supplier_accumulation['cummulative_rank'].values))
print(f"Gini coefficient for contract distribution: {gini:.4f}")

qcutoff =  0.95
qx = min(supplier_accumulation[supplier_accumulation['cummulative_rank'] >= qcutoff].index)
ncontracts_qx = supplier_accumulation['cummulative_ncontracts'].iloc[qx]

plt.figure(figsize=(10, 6))
plt.plot(supplier_accumulation['cummulative_rank'], supplier_accumulation['cummulative_ncontracts'], linestyle='-', label='Lorenz Curve')
plt.plot([0,1], [0,1], linestyle='--', color='red', label='Equal Distribution')
plt.axvline(x = qcutoff, color='grey', linestyle='--')
plt.axhline(y = ncontracts_qx, color='grey', linestyle='--')
#plt.title('Cumulative Distribution of Contracts by Supplier')
plt.xlabel('Cumulative Share of Suppliers (lowest to highest number of contracts)')
plt.ylabel('Cumulative Share of Contracts')
plt.text(qcutoff + 0.11, ncontracts_qx, f'''(Top {round((1 - qcutoff) * 100)} % suppliers, \n {round((1 - ncontracts_qx) * 100, 1)}% contracts)''', fontsize=12, ha='left', va='center')
plt.text(0.7, 0.01, f''' Gini coefficient: {gini:.4f}''', fontsize=12, ha='center', va='center', bbox=dict(facecolor='white', alpha=0.5))
plt.legend()


plt.show()

## Cosine distance distribution of top 5% suppliers' contracts

In [ ]:
# Load the contracts data
df = pd.read_feather(processed_data / 'similarity4uniformsampling.feather')#, columns=contract_level_cols)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(7, 3))  # 1 row, 2 columns

# First subplot: cosine_mean
sns.histplot(df['cosine_mean'], bins=20, ax=axes[0], stat='probability')
axes[0].set_xlabel('Mean Cosine Distance')
axes[0].set_xlim(0, 1)  # Set x-axis limits for better visibility
#add text observations with cosine_mean < 0.5
axes[0].text(0.4, 0.9, f'Observations with\nMean Cosine < 0.5:\n           {df[df['cosine_mean'] < 0.5].shape[0] / df.shape[0]:.3f}',
             transform=axes[0].transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
#how many observations have cosine_mean < 0.5
print(df[df['cosine_mean'] < 0.5].shape[0] / df.shape[0])

# Second subplot: cosine_q50
sns.histplot(df['cosine_q50'], bins=20, ax=axes[1], stat='probability')
axes[1].set_xlabel('Median Cosine Distance')
#add text observations with cosine_mean < 0.5
axes[1].text(0.4, 0.9, f'Observations with\nMedian Cosine < 0.5:\n           {df[df['cosine_q50'] < 0.5].shape[0] / df.shape[0]:.3f}',
             transform=axes[1].transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
             

plt.tight_layout()
#save figure
#plt.savefig('cosine_distance_distribution.png', dpi=300, bbox_inches='tight')

plt.show()


# Supplementary Information G: Hyperparameter selection

In [ ]:
files = list(supplementary_information_data.glob("CLASSPRIORTEST*.json"))
all = []

for filename in files:  
  
    with open(filename, "r") as f:
        data = json.load(f)
    
    all = all + data

classprior_df = pd.DataFrame(all)

In [ ]:
#CSFull
average_gain_unCSFull_list = []
average_gain_calCSFull_list = []
average_lift_unCSFull_list = []
average_lift_calCSFull_list = []

for row in tqdm(range(len(classprior_df)), desc = 'Row'):
    ### CSFull
    y_true_CSFull = classprior_df.iloc[row]['y_test_CSFull']
    unCSFull_prob = classprior_df.iloc[row]['uncalibrated_probabilities_CSFull']
    calCSFull_prob = classprior_df.iloc[row]['calibrated_probabilities_CSFull']
    ### Uncalibrated CSFull
    average_gain_unCSFull, average_lift_unCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = unCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_unCSFull_list.append(average_gain_unCSFull) #Saving
    average_lift_unCSFull_list.append(average_lift_unCSFull) #Saving


    ### Calibrated CSFull
    average_gain_calCSFull, average_lift_calCSFull = mini_ranking_evaluations(
        y_true = y_true_CSFull,
        y_pred_prob = calCSFull_prob, #PREDPROB
        top_k = len(y_true_CSFull), 
        prevalence = None)

    average_gain_calCSFull_list.append(average_gain_calCSFull) #Saving
    average_lift_calCSFull_list.append(average_lift_calCSFull) #Saving


#Include in dataset

#CSFull
classprior_df['avg_gain_unCSFull'] = average_gain_unCSFull_list
classprior_df['avg_gain_calCSFull'] = average_gain_calCSFull_list
classprior_df['avg_lift_unCSFull'] = average_lift_unCSFull_list
classprior_df['avg_lift_calCSFull'] = average_lift_calCSFull_list

#make class prior a number

classprior_df["class_prior"] = classprior_df["class_prior"].astype(float)

In [ ]:
import pandas as pd
from scipy.stats import pearsonr

#CORRELATION

results = []

# get reference rows (param_id == 0)
ref_df = classprior_df[classprior_df["class_prior"] == 0.05]

for subset in classprior_df["subset"].unique():
    
    # reference probabilities for this subset
    ref_probs = ref_df.loc[ref_df["subset"] == subset, "calibrated_probabilities_CSFull"].values
    
    if len(ref_probs) == 0:
        continue
        
    ref_probs = ref_probs[0]

    # compare with all other param_ids
    other_rows = classprior_df[(classprior_df["subset"] == subset) & (classprior_df["class_prior"] != 0)]

    for _, row in other_rows.iterrows():
        corr, pval = pearsonr(ref_probs, row["calibrated_probabilities_CSFull"])

        results.append({
            "subset": subset,
            "class_prior": row["class_prior"],
            "pearson_corr": corr,
            "p_value": pval
        })

corr_df = pd.DataFrame(results)
corr_df = corr_df[corr_df['class_prior'] != 0.05]

print(corr_df.groupby(['class_prior'])['pearson_corr'].agg(mean = 'mean', std = 'std'))

In [ ]:
#this is because I forgot to include the class prior, but i can get it from param id
# map_prov = {
#     '0': 0.05,
#     '1': 0.075,
#     '2':  0.1,
#     '3': 0.125,
#     '4': 0.15}

# classprior_df['class_prior'] = classprior_df['param_id'].astype('string').map(map_prov)


In [ ]:
import matplotlib.pyplot as plt

df2plot = (
    classprior_df[['class_prior', 'avg_gain_calCSFull', 'avg_lift_calCSFull']]
    .groupby(['class_prior'])
    .agg(
        mean_gain=('avg_gain_calCSFull', 'mean'),
        std_gain=('avg_gain_calCSFull', 'std'),
        mean_lift=('avg_lift_calCSFull', 'mean'),
        std_lift=('avg_lift_calCSFull', 'std'),
    )
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# Gain plot
axes[0].errorbar(
    x=df2plot['class_prior'],
    y=df2plot['mean_gain'],
    yerr=df2plot['std_gain'],
    fmt='o'
)
axes[0].set_xticks([0.05, 0.075, 0.1, 0.125, 0.15])
axes[0].set_ylim(0, 0.5)
axes[0].set_xlabel('Class prior')
axes[0].set_ylabel('Average gain')

# Lift plot
axes[1].errorbar(
    x=df2plot['class_prior'],
    y=df2plot['mean_lift'],
    yerr=df2plot['std_lift'],
    fmt='o'
)
axes[1].set_xticks([0.05, 0.075, 0.1, 0.125, 0.15])
axes[1].set_ylim(0, 3)
axes[1].set_xlabel('Class prior')
axes[1].set_ylabel('Average lift')

plt.tight_layout()
filename = figures_folder / 'classpriortest.png'
plt.savefig(filename, dpi=300, bbox_inches="tight")
plt.show()


# Supplementary Information H: Additional Performance Information

## H.1 Top %k metrics for all distributions

See `/scripts/main_text/evaluation.ipynb`

## H.2 Probability distributions

See `/scripts/main_text/evaluation.ipynb`

## H.3 Biased performance metrics

See `/scripts/main_text/evaluation.ipynb`

## H.4 Performance on uniform test

See `/scripts/main_text/evaluation.ipynb`

## H.5 Performance on inductive setting

In [ ]:

inductive_results = here('data/processed_data/inductive_data/results')
files = [f.name for f in inductive_results.iterdir() if f.is_file() and '1113' in f.name]

print(files)

pred_prob_dict = []
for file in tqdm(files, desc='Files'):
      with open(inductive_results / file, 'r') as f:
         d = json.load(f)
         pred_prob_dict = pred_prob_dict + d

df = pd.DataFrame(pred_prob_dict)
df['class_prior'] = np.where(df['class_prior']=='None', -1, df['class_prior'])
df['class_prior'] = df['class_prior'].astype(float)



In [ ]:
df.head()

In [ ]:
#replace model names
df['model'] = df['model'].replace({
    'hdsrf': 'HDSRF',
    'pubagging': 'PUBagging',
    })

#######IF ANALYSING YS
df['model'] = df['model'] + '-' + df['administration']

#replace columns
df.columns = df.columns.str.replace('uncalibrated', 'un', regex=False)
df.columns = df.columns.str.replace('calibrated', 'cal', regex=False)



In [ ]:
#**********************************************************
#What to plot #CSFull, CSU, YSFull
type_testset = 'YSFull'
y_test_col = f'y_test_{type_testset}'
y_prob_col = f'cal_probabilities_{type_testset}' 
thresholds2measure = list(np.round(np.arange(0.025,1.025, 0.025),3))

if type_testset == 'CSU':
    df = df[df['model'] != 'Ranking by CRI*'].reset_index()

topkmetrics_df = coordinates2plot(
        df2work = df, #df2work
        y_test_col = y_test_col,
        pred_prob_col = y_prob_col,
        ks = thresholds2measure, #thresholds to measure
        group_col = 'model',
        )

In [ ]:
grouped_df = topkmetrics_df.groupby(['model', 'topk_instances']).agg(
    #correct measures
    mean_robust_recall = ('robust_recall', 'mean'),
    std_robust_recall = ('robust_recall', 'std'),
    mean_robust_precision = ('robust_precision', 'mean'),
    std_robust_precision = ('robust_precision', 'std'),
    mean_robust_lift = ('robust_lift', 'mean'),
    std_robust_lift = ('robust_lift', 'std'),
    #biased measures
    mean_biased_recall = ('biased_recall', 'mean'),
    std_biased_recall = ('biased_recall', 'std'),
    mean_biased_precision = ('biased_precision', 'mean'),
    std_biased_precision = ('biased_precision', 'std'),
    mean_biased_lift = ('biased_lift', 'mean'),
    std_biased_lift = ('biased_lift', 'std'),
    #null models
    avg_prevalence = ('prevalence', 'mean'),
    mean_null_recall = ('null_recall', 'mean')
)
grouped_df = grouped_df.reset_index()
grouped_df

In [ ]:
import matplotlib.pyplot as plt

#******************************
# Loops

administration = 'EPN'

algorithms = [ 'PUBagging', 'HDSRF', 'Ranking by CRI*']
measures = ['recall', 'precision', 'lift']
type_measure = 'robust'
figsize = (10,5)

if type_testset in ['CSU']:
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)

if type_testset in ['YSFull']:
    algorithms = [ f'PUBagging-{administration}', f'HDSRF-{administration}']
    figsize = (7.5,5)


#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['model', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('model').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.02, 0.05),
    'lift': (0.02, 2)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['model'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.25)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 11)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

plt.show()

In [ ]:
import matplotlib.pyplot as plt

#******************************
# Loops

administration = 'AMLO'

algorithms = [ 'PUBagging', 'HDSRF', 'Ranking by CRI*']
measures = ['recall', 'precision', 'lift']
type_measure = 'robust'
figsize = (10,5)

if type_testset in ['CSU']:
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)

if type_testset in ['YSFull']:
    algorithms = [ f'PUBagging-{administration}', f'HDSRF-{administration}']
    figsize = (7.5,5)


#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['model', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('model').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.02, 0.05),
    'lift': (0.02, 2)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['model'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.25)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 11)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

plt.show()

## H.6. Random Permutation Testing

In [ ]:
df_perm = pd.read_feather(processed_data / 'metrics_permutation_testing.feather',
                     columns= [
                        'random_state', 
                        'subset', 
                        'avg_gain_calCSFull',
                        'avg_lift_calCSFull', 
                        'map_calCSFull'
                        ])

df_perm = df_perm.dropna().reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ------------------- SETTINGS -------------------
fontsize = 9

# palette
lower_blue = '#40798C'
red = '#ef476f'

plt.style.use('ggplot')

# ------------------- PERMUTATION TESTING PLOT -------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
axes[1].tick_params(left=False, labelleft=False)

metrics = [
    ('avg_gain_calCSFull', r'$\overline{Gain}$'),
    ('avg_lift_calCSFull', r'$\overline{L_c}$')
]

for idx, (ax, (metric_prefix, metric_label)) in enumerate(zip(axes, metrics)):
    x = df_perm[f'{metric_prefix}']

    if metric_prefix == 'avg_gain_calCSFull':
        my_model_value = 0.31
    else:
        my_model_value = 2.31

    ax.hist(
        x,
        bins=50,
        alpha=0.5,
        color=red,
        density=False,
        label='Permutations'
    )

    ax.axvline(
        x=my_model_value,
        color=lower_blue,
        linestyle='--',
        label='Our HDSRF'
    )

    if idx == 0:
        ax.set_ylabel('Frequency', fontsize=fontsize)
        ax.legend(fontsize=fontsize)
    else:
        ax.set_ylabel('')

    if idx == 1:
        ax.set_xlim(-0.01, 2.6)

    ax.set_xlabel(metric_label, fontsize=fontsize, labelpad=0)

plt.tight_layout()

plt.show()

# Supplementary Information I: Additional Model Interpretation Information

## I.1 Performance metrics for network and domain knowledge

In [ ]:

######################################################################
results_folder = here() / 'data' / 'processed_data' / 'suplementary_data'
files = list(supplementary_information_data.glob("ABLATIONTEST*.json"))
all = []

for filename in files:  
  
    with open(filename, "r") as f:
        data = json.load(f)
    
    all = all + data

df = pd.DataFrame(all)

df['feature2keep'] = df['feature2keep'].str.upper()

print(df.columns)


In [ ]:

#**********************************************************
#What to plot
y_test_col = 'y_test_CSFull'
y_prob_col = 'calibrated_probabilities_CSFull' 
thresholds2measure = list(np.round(np.arange(0.025,1.025, 0.025),3))

topkmetrics_df = coordinates2plot(
        df2work = df, #df2work
        y_test_col = y_test_col,
        pred_prob_col = y_prob_col,
        ks = thresholds2measure, #thresholds to measure
        group_col = 'feature2keep',
        )

In [ ]:
grouped_df = topkmetrics_df.groupby(['feature2keep', 'topk_instances']).agg(
    #correct measures
    mean_robust_recall = ('robust_recall', 'mean'),
    std_robust_recall = ('robust_recall', 'std'),
    mean_robust_precision = ('robust_precision', 'mean'),
    std_robust_precision = ('robust_precision', 'std'),
    mean_robust_lift = ('robust_lift', 'mean'),
    std_robust_lift = ('robust_lift', 'std'),
    #biased measures
    mean_biased_recall = ('biased_recall', 'mean'),
    std_biased_recall = ('biased_recall', 'std'),
    mean_biased_precision = ('biased_precision', 'mean'),
    std_biased_precision = ('biased_precision', 'std'),
    mean_biased_lift = ('biased_lift', 'mean'),
    std_biased_lift = ('biased_lift', 'std'),
    #null models
    avg_prevalence = ('prevalence', 'mean'),
    mean_null_recall = ('null_recall', 'mean')
)
grouped_df = grouped_df.reset_index()


In [ ]:
import matplotlib.pyplot as plt

#******************************
# Loops

algorithms = [ 'DOMAIN_KNOWLEDGE', 'NETWORK']
measures = ['recall', 'precision', 'lift']
type_measure = 'robust'
figsize = (10,6)
type_testset = 'CSFull'

if type_testset in ['CSU']:
    algorithms = [ 'PUBagging', 'HDSRF']
    figsize = (7.5,5)

if type_testset in ['YSFull']:
    algorithms = [ 'PUBagging-EPN', 'HDSRF-EPN']
    figsize = (7.5,5)


#******************************
#Aestetics

plt.style.use('ggplot')
lower_blue = '#40798C'
blue = '#2b9eb3'

#******************************
#Notes
notes = topkmetrics_df[['feature2keep', f'{type_measure}_average_gain', f'{type_measure}_average_precision', f'{type_measure}_average_lift']].drop_duplicates().groupby('feature2keep').agg(('mean', 'std')).round(2).to_dict()
#Label position
maxis = {
    'recall':(0.7, 0.25),
    'precision':(0.20, 0.15),
    'lift': (0.20, 4)
}
#******************************
#Axis labels
if type_measure == 'robust':
    measures_labels = [
        r'$R_{R}@k$',
        r'$P_{R}@k$',
        r'$L_{R}@k$'
    ]
else:
    measures_labels = [
        r'$R@k$',
        r'$P@k$',
        r'$L@k$'
    ]
measures_labels_dict = dict(zip(measures, measures_labels))
#******************************
fig, axs = plt.subplots(
    3,len(algorithms),
    gridspec_kw={'height_ratios': [2, 1, 1]},
    figsize=figsize)

for col, algorithm in enumerate(algorithms):

    df2plot = grouped_df[grouped_df['feature2keep'] == algorithm]

    for row, measure in enumerate(measures):

        pointsize = 3 if row == 0 else 6
        if row == 0:
            avg_n = notes[(f'{type_measure}_average_gain', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_gain', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{Gain}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 1:
            avg_n = notes[(f'{type_measure}_average_precision', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_precision', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{P_R}$: ' + f'\n {avg_n}  ±  {std_n}'
        if row == 2:
            avg_n = notes[(f'{type_measure}_average_lift', 'mean')][f'{algorithm}']
            std_n = notes[(f'{type_measure}_average_lift', 'std')][f'{algorithm}']
            textonplot = r'        $\overline{L_R}$: ' + f'\n {avg_n}  ±  {std_n}'

        ax = axs[row, col]

        ax.errorbar(
            x=df2plot['topk_instances'],
            y=df2plot[f'mean_{type_measure}_{measure}'],
            color=lower_blue,
            label = 'Our model',
            marker = 'o',
            markersize = pointsize,
            
        )

        if row != 1:
            ax.text(
                maxis[measure][0],
                maxis[measure][1],
                textonplot,
                fontsize=6,
                verticalalignment='bottom'
                )
            #print(textonplot)

        if measure == 'recall':
            ax.fill_between(
                x = df2plot['topk_instances'],
                y1 = df2plot['mean_null_recall'],
                y2 = df2plot[f'mean_{type_measure}_{measure}'],
                #step='mid',
                alpha=0.8,
                color=blue,
                label='Gain',
                linestyle = '-'
            )


            ax.plot(
                df2plot['topk_instances'],
                df2plot['mean_null_recall'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )

        elif measure == 'precision':
            ax.plot(
                df2plot['topk_instances'],
                df2plot['avg_prevalence'],
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-0.05, 0.25)
            ax.set_xlim(0, 0.275)


        
        elif measure == 'lift':
            ax.plot(
                df2plot['topk_instances'],
                np.repeat([1], len(df2plot['topk_instances'])),
                linestyle='--',
                color='red',
                alpha=0.5,
                label = 'Null Model'
            )
            ax.set_ylim(-1, 8)
            ax.set_xlim(0, 0.275)


        ax.tick_params(axis='x', rotation=0, labelsize = 6)
        ax.tick_params(axis='y', rotation=0, labelsize = 6)


        if col == 0:
            ax.set_ylabel(measures_labels_dict[measure], fontsize = 9)
            if row == 0:
                ax.legend(loc='upper left', fontsize=8)
        else:
            ax.set_yticklabels('')

        if row == 0:
            ax.set_title(algorithm, fontsize = 9)

        if row == 1:
            ax.set_xticklabels('')
        if (row == 2) & ( col == 1 ):
            ax.set_xlabel('Top k instances', fontsize = 9)
            #
        
# reduce whitespace between plots
fig.subplots_adjust(hspace=0.2, wspace=0.05)    

#plt.tight_layout()
filename = f'evaluation_ablation.png'
plt.savefig(filename, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cols = [
    "feature2keep", "topk_instances",
    "mean_robust_recall", "std_robust_recall",
    "mean_robust_precision", "std_robust_precision",
    "mean_robust_lift", "std_robust_lift"
]

df_sub = grouped_df[cols].copy()
def fmt(mean, std):
    return mean.round(3).astype(str) + " (" + std.round(3).astype(str) + ")"

df_sub["robust_recall"] = fmt(df_sub["mean_robust_recall"], df_sub["std_robust_recall"])
df_sub["robust_precision"] = fmt(df_sub["mean_robust_precision"], df_sub["std_robust_precision"])
df_sub["robust_lift"] = fmt(df_sub["mean_robust_lift"], df_sub["std_robust_lift"])
df_sub = df_sub[["feature2keep", "topk_instances",
                 "robust_recall", "robust_precision", "robust_lift"]]
df_pivot = df_sub.pivot(index="topk_instances", columns="feature2keep")
df_pivot.columns = [
    f"{metric}_{model}"
    for metric, model in df_pivot.columns
]


In [ ]:
#to latex
latex_str = df_pivot.to_latex(index=True)
print(latex_str)

## I.2 Additional Shap Plots

See `/scripts/main_text/shap_interpretation.ipynb`